# Data Pipeline: Setup, Load, Validate, Create Loaders

**Purpose**: Single-run notebook that prepares the full data pipeline for downstream notebooks.

**Outputs**:
- Dataset loaded and cached
- Data validated (quality checks)
- PyTorch DataLoaders ready for training

**Run this first**, then other notebooks can import loaders.

**Issues**: #3 Data Loading (D1), #4 Exploratory Analysis (D1), #6 Data Cleaning (D2)

## Setup: Clone Repository & Install Dependencies

In [ ]:
import os
import sys

REPO_DIR = '/content/RETINNA'
if os.path.exists(REPO_DIR):
    print("Updating repository...")
    os.chdir(REPO_DIR)
    os.system('git pull origin main')
else:
    print("Cloning RETINNA repository...")
    os.system('git clone https://github.com/scerruti/RETINNA.git /content/RETINNA')
    os.chdir(REPO_DIR)

print(f"✓ Repository ready at {REPO_DIR}")
%pip install -q torch torchvision matplotlib numpy scipy scikit-learn torchgeo

src_path = '/content/RETINNA/src'
if src_path not in sys.path:
    sys.path.append(src_path)

from colab_utils import setup_colab_environment, setup_cabuaur_cached

env = setup_colab_environment()
cabuaur_path = setup_cabuaur_cached(env)

print("✓ Environment ready")

## Load Dataset

In [ ]:
from torchgeo.datasets import CaBuAr

print("Loading CaBuAr dataset...")
dataset = CaBuAr(root=cabuaur_path, download=True, split='test')
print(f"✓ Loaded {len(dataset)} samples from {cabuaur_path}")

sample = dataset[0]
print(f"\nSample structure:")
for key, value in sample.items():
    if hasattr(value, 'shape'):
        print(f"  {key}: shape {tuple(value.shape)}, dtype {value.dtype}")
    else:
        print(f"  {key}: {type(value).__name__}")

## Validate Data Quality

In [ ]:
from data_cleaning import validate_dataset

results = validate_dataset(dataset_root=cabuaur_path)

print("\n✅ Validation complete")

## Create PyTorch DataLoaders

In [ ]:
from dataset import get_dataloaders

print("Creating PyTorch DataLoaders...\n")

dataloaders = get_dataloaders(batch_size=4, num_workers=0, root=cabuaur_path)

# Test one batch
train_loader = dataloaders['train']
batch = next(iter(train_loader))

print(f"Batch loaded successfully:")
print(f"  Images: {batch['image'].shape}")
print(f"  Masks: {batch['mask'].shape}")
print(f"\n✅ DataLoaders ready for training")

## Summary

In [ ]:
print("\n" + "="*70)
print("✅ DATA PIPELINE READY")
print("="*70)
print(f"""
Next steps:
  1. Exploratory analysis: notebooks/02_exploratory_analysis.ipynb
  2. Training: notebooks/03_training.ipynb (coming soon)
  3. Inference: notebooks/04_inference.ipynb (coming soon)

DataLoaders available as:
  dataloaders['train']  - Training data
  dataloaders['val']    - Validation data
  dataloaders['test']   - Test data
  dataloaders['datasets'] - Raw dataset objects
""")
print("="*70)